# 00 — Check your OpenMC setup

Run all cells (**Run All** at the top). If the last cell prints a k-effective value, your environment works.

In [ ]:
import os, openmc
print('OpenMC version:', openmc.__version__)
print('Cross sections:', os.environ.get('OPENMC_CROSS_SECTIONS'))

## A simple UO$_2$ pincell

In [ ]:
# Materials
fuel = openmc.Material(name='UO2 (3.1 wt%)')
fuel.add_element('U', 1.0, enrichment=3.1)
fuel.add_element('O', 2.0)
fuel.set_density('g/cm3', 10.4)

clad = openmc.Material(name='Zircaloy')
clad.add_element('Zr', 1.0)
clad.set_density('g/cm3', 6.55)

water = openmc.Material(name='Water')
water.add_element('H', 2.0)
water.add_element('O', 1.0)
water.set_density('g/cm3', 0.74)
water.add_s_alpha_beta('c_H_in_H2O')

materials = openmc.Materials([fuel, clad, water])

# Geometry: fuel pin in a reflective square cell
fuel_or = openmc.ZCylinder(r=0.39)
clad_or = openmc.ZCylinder(r=0.46)
pitch = 1.26
box = openmc.model.RectangularPrism(pitch, pitch, boundary_type='reflective')

cells = [
    openmc.Cell(fill=fuel, region=-fuel_or),
    openmc.Cell(fill=clad, region=+fuel_or & -clad_or),
    openmc.Cell(fill=water, region=+clad_or & -box),
]
geometry = openmc.Geometry(cells)

# Settings
settings = openmc.Settings()
settings.batches = 50
settings.inactive = 10
settings.particles = 2000
settings.source = openmc.IndependentSource(space=openmc.stats.Point())

model = openmc.Model(geometry, materials, settings)

In [ ]:
geometry.plot(basis='xy', colors={fuel: 'firebrick', clad: 'lightgray', water: 'dodgerblue'}, color_by='material')

In [ ]:
sp_path = model.run(output=False)
with openmc.StatePoint(sp_path) as sp:
    print('k-effective =', sp.keff)